# KOVA3 tutorial 2: annotate your own VCF with Korean allele frequencies

**KOVA3 release:** `v3.0.0` (pinned; see the release manifest)
**Tier:** Open (no credentials, no AWS account required)
**Tools:** bcftools 1.19+ (htslib built with libcurl), a GRCh38 reference FASTA for indel left-alignment

This is the most common thing people do with a population frequency resource: take a patient or cohort VCF and ask, for every variant in it, how often that allele is seen in Koreans. You then filter on that number.

Two things make this go wrong in practice, and both are handled below.

1. **Field-name collision.** KOVA3 publishes its frequencies under the community-standard names `AC`, `AN` and `AF`. Your own VCF almost certainly has fields with exactly those names describing *your* cohort. Annotating without renaming silently overwrites yours.
2. **Normalization mismatch.** A multi-allelic record on one side and split records on the other will not join. A differently left-aligned indel joins to the wrong row, or to none.

> **Draft note.** The `kova3-open` bucket does not exist yet, so the cells that read from S3 carry no outputs. They will be run and the notebook republished with outputs at release.

## 0. Parameters

In [ ]:
# Open-tier bucket and the release this notebook is pinned to
BUCKET  = "kova3-open"
REGION  = "ap-northeast-2"
RELEASE = "v3.0.0"
BASE    = f"https://{BUCKET}.s3.{REGION}.amazonaws.com/data/release={RELEASE}/sites_vcf"

# The chromosome we work on in this example
CHROM   = "chr17"
KOVA3_VCF = f"{BASE}/kova3.{CHROM}.sites.vcf.gz"

# A GRCh38 reference FASTA, needed to left-align indels. Use the same assembly
# KOVA3 is on; any GRCh38 analysis-set FASTA works as long as you use the same
# one on both sides of the comparison. Leave as None to skip the indel step.
REF_FASTA = None

print(KOVA3_VCF)

## 1. A small example VCF to annotate

So that the notebook is self-contained we write a five-variant VCF in *BRCA1*. Substitute your own file for `patient.vcf.gz` and everything below is unchanged.

Note that this example VCF declares its own `AC`, `AN` and `AF`. That is the point: it is what makes the collision visible.

In [ ]:
import subprocess, textwrap, os

PATIENT_VCF = "patient.vcf.gz"

vcf = textwrap.dedent("""\
##fileformat=VCFv4.2
##contig=<ID=chr17,length=83257441>
##INFO=<ID=AC,Number=A,Type=Integer,Description="Allele count in this cohort">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Allele number in this cohort">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele frequency in this cohort">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr17\t43045711\t.\tT\tC\t500\tPASS\tAC=1;AN=200;AF=0.005
chr17\t43047643\t.\tA\tG\t500\tPASS\tAC=3;AN=200;AF=0.015
chr17\t43063903\t.\tC\tT\t500\tPASS\tAC=2;AN=200;AF=0.010
chr17\t43082434\t.\tG\tA\t500\tPASS\tAC=8;AN=200;AF=0.040
chr17\t43093464\t.\tA\tG\t500\tPASS\tAC=1;AN=200;AF=0.005
""")

with open("patient.vcf", "w") as fh:
    fh.write(vcf)

subprocess.run(["bcftools", "view", "-Oz", "-o", PATIENT_VCF, "patient.vcf"], check=True)
subprocess.run(["bcftools", "index", "-t", PATIENT_VCF], check=True)
print(subprocess.run(["bcftools", "query", "-f", "%CHROM\t%POS\t%REF\t%ALT\t%INFO/AF\n", PATIENT_VCF],
                     capture_output=True, text=True, check=True).stdout)

## 2. Normalize both sides the same way

KOVA3 is published with multi-allelic sites already split and indels left-aligned, one ALT per row. Your VCF must match, or the join is silently wrong for exactly the variants you care most about.

`-m -any` splits; `-f` supplies the reference that left-alignment needs. Run this even if you believe your file is already normalized: it is cheap and it is not detectable afterwards when it was skipped.

In [ ]:
NORM_VCF = "patient.norm.vcf.gz"

cmd = ["bcftools", "norm", "-m", "-any"]
if REF_FASTA:
    cmd += ["-f", REF_FASTA]
else:
    print("REF_FASTA is None: splitting multi-allelics only, without indel left-alignment.\n"
          "That is fine for the SNV example below, but set REF_FASTA before using this on real data.")
cmd += ["-Oz", "-o", NORM_VCF, PATIENT_VCF]

subprocess.run(cmd, check=True)
subprocess.run(["bcftools", "index", "-t", NORM_VCF], check=True)
print("normalized ->", NORM_VCF)

## 3. Annotate, with the fields renamed on the way in

This is the whole tutorial in one command. The `:=` operator renames each KOVA3 field as it is written, so `AC` arrives as `KOVA3_AC` and your own `AC` survives untouched.

Only the compressed blocks covering your variants are fetched from S3; the shard is not downloaded.

In [ ]:
ANNOTATED = "patient.kova3.vcf.gz"

subprocess.run([
    "bcftools", "annotate",
    "-a", KOVA3_VCF,
    "-c", ("INFO/KOVA3_AC:=INFO/AC,"
           "INFO/KOVA3_AN:=INFO/AN,"
           "INFO/KOVA3_AF:=INFO/AF,"
           "INFO/KOVA3_nhomalt:=INFO/nhomalt,"
           "INFO/KOVA3_call_rate:=INFO/call_rate"),
    "-Oz", "-o", ANNOTATED, NORM_VCF,
], check=True)
subprocess.run(["bcftools", "index", "-t", ANNOTATED], check=True)

out = subprocess.run([
    "bcftools", "query",
    "-f", "%CHROM\t%POS\t%REF\t%ALT\t%INFO/AF\t%INFO/KOVA3_AF\t%INFO/KOVA3_AN\t%INFO/KOVA3_nhomalt\n",
    ANNOTATED,
], capture_output=True, text=True, check=True).stdout

print("CHROM\tPOS\tREF\tALT\tyour_AF\tKOVA3_AF\tKOVA3_AN\tKOVA3_nhomalt")
print(out)

**What the columns mean.** `your_AF` is untouched, which is the point of the rename. `KOVA3_AF` is the Korean frequency. `KOVA3_AN` is the denominator behind it, and you need it: with 11,000 genomes `AN` reaches 22,000 on the autosomes, so a low `KOVA3_AF` with an `AN` of 21,000 is well-powered evidence, and the same frequency with an `AN` of 400 is not evidence of anything.

A variant with no KOVA3 annotation at all (`.`) was not in the callset. That is not the same as being absent from Koreans: the site may not have been callable. Tutorial 1 and the `callability/` resources cover how to tell the difference.

## 4. What the rename is protecting you from

Run the same annotation without `:=` and compare. This overwrites your cohort's own numbers with Korean ones, and nothing in the output says it happened. It is worth seeing once.

In [ ]:
subprocess.run([
    "bcftools", "annotate", "-a", KOVA3_VCF,
    "-c", "INFO/AC,INFO/AN,INFO/AF",       # no renaming: destructive
    "-Oz", "-o", "patient.clobbered.vcf.gz", NORM_VCF,
], check=True)

before = subprocess.run(["bcftools", "query", "-f", "%POS\t%INFO/AF\n", NORM_VCF],
                        capture_output=True, text=True, check=True).stdout
after = subprocess.run(["bcftools", "query", "-f", "%POS\t%INFO/AF\n", "patient.clobbered.vcf.gz"],
                       capture_output=True, text=True, check=True).stdout

print("AF before annotation (your cohort):")
print(before)
print("AF after annotation without renaming (now Korean, silently):")
print(after)

The same applies to every other frequency resource you annotate alongside KOVA3. Give each one its own prefix, or the last one written wins.

## 5. Filter on the Korean frequency

Now the actual task. Keep variants that are rare in Koreans, and keep sites where KOVA3 has no call at all rather than dropping them silently.

The `KOVA3_AN` condition is what stops a poorly covered site from being mistaken for a rare one.

In [ ]:
RARE = "patient.kova3.rare.vcf.gz"

EXPR = ('INFO/KOVA3_AF < 0.01 && INFO/KOVA3_AN > 15000')   # rare and well powered
KEEP_UNSEEN = ('INFO/KOVA3_AF = "."')                      # not in the callset

subprocess.run([
    "bcftools", "view", "-i", f"({EXPR}) || ({KEEP_UNSEEN})",
    "-Oz", "-o", RARE, ANNOTATED,
], check=True)

n_in = subprocess.run(["bcftools", "index", "-n", ANNOTATED], capture_output=True, text=True).stdout.strip()
subprocess.run(["bcftools", "index", "-t", RARE], check=True)
n_out = subprocess.run(["bcftools", "index", "-n", RARE], capture_output=True, text=True).stdout.strip()
print(f"{n_in} variants in, {n_out} retained after the Korean frequency filter")

**Choosing the threshold.** `AN > 15000` is a call rate of roughly 0.68 across the cohort. Raise it if your analysis needs more power at the cost of coverage, lower it if you would rather keep marginal sites and inspect them. There is nothing special about 15,000 beyond it being a reasonable default for this cohort size.

For ACMG/AMP work, BA1 and BS1 are frequency thresholds set by the disorder, not by the resource. KOVA3 gives you the population-matched frequency to apply them to; the threshold itself comes from your disease model.

## 6. The Jeju stratum, if you need it

KOVA3 publishes one subpopulation stratum, the Jeju Genome cohort, as `AC_jeju`, `AN_jeju`, `AF_jeju` and `nhomalt_jeju`. It is defined by cohort of origin, not by inferred ancestry.

Use the whole-cohort `AF` unless your samples are from Jeju or you are specifically asking whether a frequency is uniform across Korea. The stratum denominator is roughly a quarter of the whole-cohort one, so estimates from it carry correspondingly more uncertainty. Check `AN_jeju` before drawing any conclusion from `AF_jeju`.

In [ ]:
subprocess.run([
    "bcftools", "annotate", "-a", KOVA3_VCF,
    "-c", ("INFO/KOVA3_AF:=INFO/AF,INFO/KOVA3_AN:=INFO/AN,"
           "INFO/KOVA3_AF_jeju:=INFO/AF_jeju,INFO/KOVA3_AN_jeju:=INFO/AN_jeju"),
    "-Oz", "-o", "patient.kova3.jeju.vcf.gz", NORM_VCF,
], check=True)

out = subprocess.run([
    "bcftools", "query",
    "-f", "%POS\t%INFO/KOVA3_AF\t%INFO/KOVA3_AN\t%INFO/KOVA3_AF_jeju\t%INFO/KOVA3_AN_jeju\n",
    "patient.kova3.jeju.vcf.gz",
], capture_output=True, text=True, check=True).stdout

print("POS\tAF_all\tAN_all\tAF_jeju\tAN_jeju")
print(out)

## Expected output (to be filled at release)

| Item | Value |
|---|---|
| Release | `v3.0.0` |
| Input | 5 SNVs in *BRCA1* on chr17 |
| Variants annotated | `<N>` of 5 |
| Runtime | `<~3-8 s>` on a typical connection |
| Bytes transferred | `<~1-2 MB>` (index plus the overlapping blocks) |
| User-side AWS cost | none (public bucket, sponsored egress) |

## Next steps

Tutorial 3 runs the same lookup against the Parquet layer with Amazon Athena, which is the cheaper route for a gene panel or a list of a few thousand variants. See the [tutorials index](README.md).